In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Data Ingestion

In [2]:
orders = pd.read_csv("C:/Users/Suraj/Downloads/Olist_Project/olist_orders_dataset.csv")
order_items = pd.read_csv("C:/Users/Suraj/Downloads/Olist_Project/olist_order_items_dataset.csv")
payments = pd.read_csv("C:/Users/Suraj/Downloads/Olist_Project/olist_order_payments_dataset.csv")
reviews = pd.read_csv("C:/Users/Suraj/Downloads/Olist_Project/olist_order_reviews_dataset.csv")
customers = pd.read_csv("C:/Users/Suraj/Downloads/Olist_Project/olist_customers_dataset.csv")
sellers = pd.read_csv("C:/Users/Suraj/Downloads/Olist_Project/olist_sellers_dataset.csv")
products = pd.read_csv("C:/Users/Suraj/Downloads/Olist_Project/olist_products_dataset.csv")
geolocation = pd.read_csv("C:/Users/Suraj/Downloads/Olist_Project/olist_geolocation_dataset.csv")
categories = pd.read_csv("C:/Users/Suraj/Downloads/Olist_Project/product_category_name_translation.csv")

In [3]:
dfs = {
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "customers": customers,
    "sellers": sellers,
    "products": products,
    "geolocation": geolocation,
    "categories": categories
}

In [4]:
for name, df in dfs.items():
    print(f"\n{name.upper()} DATASET")
    print(df.info())


ORDERS DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB
None

ORDER_ITEMS DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   ord

In [7]:
def clean_data(name, df):
    
    # Remove duplicates
    df = df.drop_duplicates()
    
    # Standardize column names
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    
    # Dataset-specific transformations
    if name == "orders" and "order_purchase_timestamp" in df.columns:
        df["order_purchase_timestamp"] = pd.to_datetime(
            df["order_purchase_timestamp"], errors="coerce"
        )
    
    elif name == "payments" and "payment_value" in df.columns:
        df = df[df["payment_value"] > 0].copy()
    
    # Handle missing values (object columns)
    for col in df.select_dtypes(include="object").columns:
        df.loc[:,col] = df[col].fillna("Unknown")
    
    # Handle missing values (numeric columns)
    for col in df.select_dtypes(include=["float", "int"]).columns:
        df.loc[:,col] = df[col].fillna(0)
    
    return df

In [8]:
for name in dfs:
    dfs[name] = clean_data(name, dfs[name])

In [9]:
print(dfs["orders"].info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99441 non-null  object        
 5   order_delivered_carrier_date   99441 non-null  object        
 6   order_delivered_customer_date  99441 non-null  object        
 7   order_estimated_delivery_date  99441 non-null  object        
dtypes: datetime64[ns](1), object(7)
memory usage: 6.1+ MB
None


In [10]:
for name, df in dfs.items():
    df.to_csv(f"C:/Users/Suraj/Desktop/projects/Olist_Project/data/cleaned/{name}_cleaned.csv", index=False)

## Building Relational Database Schema from Cleaned Datasets Using Python

In [19]:
# -------------------------------
# STEP 1: Define dtype mapping
# -------------------------------
def map_dtype(dtype):
    dtype = str(dtype)
    
    if "int" in dtype:
        return "INT"
    elif "float" in dtype:
        return "DECIMAL(10,2)"
    elif "datetime" in dtype:
        return "DATETIME"
    else:
        return "VARCHAR(100)"


# -------------------------------
# STEP 2: Define Primary Keys
# -------------------------------
primary_keys = {
    "orders": ["order_id"],
    "customers": ["customer_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "order_items": ["order_id", "product_id"]  # composite key
}


# -------------------------------
# STEP 3: Define Foreign Keys
# -------------------------------
# foreign_keys = {
#     "orders": [
#         ("customer_id", "customers", "customer_id")
#     ],
#     "order_items": [
#         ("order_id", "orders", "order_id"),
#         ("product_id", "products", "product_id")
#     ],
#     "payments": [
#         ("order_id", "orders", "order_id")
#     ],
#     "reviews": [
#         ("order_id", "orders", "order_id")
#     ]
# }


# -------------------------------
# STEP 4: Generate SQL Schema
# -------------------------------
all_tables_sql = ""

for table_name, df in dfs.items():
    
    columns_sql = []
    
    # Columns with datatypes
    for col, dtype in zip(df.columns, df.dtypes):
        sql_type = map_dtype(dtype)
        columns_sql.append(f"{col} {sql_type}")
    
    # Add PRIMARY KEY
    pk = primary_keys.get(table_name)
    if pk:
        columns_sql.append(f"PRIMARY KEY ({', '.join(pk)})")
    
    # Add FOREIGN KEYS
    # fks = foreign_keys.get(table_name, [])
    # for col, ref_table, ref_col in fks:
    #     columns_sql.append(
    #         f"FOREIGN KEY ({col}) REFERENCES {ref_table}({ref_col})"
    #     )
    
    # Create table statement
    columns_formatted = ',\n    '.join(columns_sql)

    create_statement = f"""
    CREATE TABLE {table_name} (
        {columns_formatted}
    );
    """
    
    all_tables_sql += create_statement + "\n"


# -------------------------------
# STEP 5: Save to schema.sql
# -------------------------------
with open("C:/Users/Suraj/Desktop/projects/Olist_Project/sql/schema.sql", "w") as f:
    f.write(all_tables_sql)

print("schema.sql file generated successfully!")

schema.sql file generated successfully!
